# 2.03 贝尔曼最优方程：什么是"最好的策略"

[![在 Colab 中打开](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ROI-su7/vla-ladder/blob/main/notebooks/02_rl/2.03_bellman_optimality.ipynb)


## 为什么要学

上一课学会了**评价**策略，这一课回答：**最好的策略是什么？怎么找到它？**

答案就藏在一个方程里，即贝尔曼最优方程（Bellman Optimality Equation，BOE）。Q-learning、DQN 的 TD 目标 $r+\gamma\max_{a'}Q(s',a')$ 都直接来自它。对应书中第 3 章。

---

## 1. 最优策略

如果策略 $\pi^*$ 满足：对**所有**状态 $s$ 和**所有**其它策略 $\pi$，都有 $v_{\pi^*}(s)\ge v_\pi(s)$，那么它就是**最优策略**。

问题：这样的策略存在吗？唯一吗？怎么求？BOE 回答了这三个问题。

## 2. 贝尔曼最优方程

$$
v(s) = \max_{\pi}\sum_a\pi(a|s)\underbrace{\left[\sum_r p(r|s,a)r + \gamma\sum_{s'}p(s'|s,a)v(s')\right]}_{q(s,a)} = \max_a q(s,a)
$$

为什么 $\max_\pi\sum_a\pi(a|s)q(s,a) = \max_a q(s,a)$？因为加权平均不可能超过最大值，把全部概率放在 $q$ 最大的动作上就达到了最大值。这说明：

> **最优策略可以是确定性的贪心策略**：$\pi^*(s) = \arg\max_a q^*(s,a)$。

矩阵形式：

$$
\mathbf{v} = \max_\pi(\mathbf{r}_\pi + \gamma P_\pi\mathbf{v}) \triangleq f(\mathbf{v})
$$

这是一个**非线性**方程（因为有 max）。怎么解？

## 3. 压缩映射定理

如果函数 $f$ 满足 $\|f(\mathbf{x}_1)-f(\mathbf{x}_2)\|\le\gamma\|\mathbf{x}_1-\mathbf{x}_2\|$（$\gamma<1$），就称它为**压缩映射**。压缩映射定理（Banach 不动点定理）保证：

1. **存在性**：存在不动点 $\mathbf{x}^*$ 使 $f(\mathbf{x}^*)=\mathbf{x}^*$；
2. **唯一性**：不动点唯一；
3. **算法**：从任意 $\mathbf{x}_0$ 出发迭代 $\mathbf{x}_{k+1}=f(\mathbf{x}_k)$，以指数速度收敛到 $\mathbf{x}^*$。

书中证明了 BOE 右边的 $f(\mathbf{v})$ 在 $\|\cdot\|_\infty$ 下是压缩映射（压缩系数就是 $\gamma$）。所以：

- BOE 的解 $\mathbf{v}^*$ **存在且唯一**；
- 迭代 $\mathbf{v}_{k+1} = f(\mathbf{v}_k)$ 一定收敛到它：这就是**值迭代**；
- 对 $\mathbf{v}^*$ 取贪心得到的策略 $\pi^*$ 就是最优策略，且 $\mathbf{v}^* = \mathbf{v}_{\pi^*}$。

（注意：最优**值**唯一，但最优**策略**可能不唯一，比如两个动作的 $q$ 一样大时。）

---

## 4. 代码：求解 BOE

In [ ]:
# 🔧 环境准备（每个 notebook 都先运行这一格）
# - 在 Colab 中：自动下载本项目代码
# - 在本地：自动把项目根目录加入 sys.path（也可以在根目录执行 `pip install -e .`）
import os, sys
if "google.colab" in sys.modules:
    if not os.path.exists("/content/vla-ladder"):
        os.system("git clone -q --depth 1 https://github.com/ROI-su7/vla-ladder.git /content/vla-ladder")
    sys.path.insert(0, "/content/vla-ladder")
else:
    _p = os.path.abspath(".")
    while _p != os.path.dirname(_p) and not os.path.isdir(os.path.join(_p, "vla_ladder")):
        _p = os.path.dirname(_p)
    sys.path.insert(0, _p)
print("环境就绪 ✅")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from vla_ladder.envs import GridWorld, policy_to_matrix

env = GridWorld()

### 练习 1：贝尔曼最优算子与值迭代

In [ ]:
def bellman_optimality_operator(env, v, gamma):
    """f(v)：返回 (新的 v (S,), 贪心策略 (S,))"""
    # TODO: q = R + γ P @ v，形状 (S, A)；新 v = q.max(1)；贪心策略 = q.argmax(1)
    raise NotImplementedError

def value_iteration(env, gamma, tol=1e-6, max_iter=10000):
    """返回 (v*, π*, 每次迭代的 ||v_{k+1} - v_k||_∞ 列表)"""
    v = np.zeros(env.n_states)
    deltas = []
    # TODO: 反复调用 bellman_optimality_operator，直到变化量 < tol
    raise NotImplementedError

In [ ]:
#@title 💡 参考答案（点击展开查看；运行本单元即可使用参考实现） { display-mode: "form" }
def bellman_optimality_operator(env, v, gamma):
    """f(v)：返回 (新的 v (S,), 贪心策略 (S,))"""
    q = env.R + gamma * env.P @ v      # (S, A)
    return q.max(axis=1), q.argmax(axis=1)

def value_iteration(env, gamma, tol=1e-6, max_iter=10000):
    """返回 (v*, π*, 每次迭代的 ||v_{k+1} - v_k||_∞ 列表)"""
    v = np.zeros(env.n_states)
    deltas = []
    for _ in range(max_iter):
        v_new, pi = bellman_optimality_operator(env, v, gamma)
        deltas.append(np.max(np.abs(v_new - v)))
        v = v_new
        if deltas[-1] < tol:
            break
    return v, pi, deltas

In [ ]:
# ✅ 自测
gamma = 0.9
v_star, pi_star, deltas = value_iteration(env, gamma)
# 最优策略的值 = BOE 的解：用上一课的闭式解策略评估验证
r_pi = (policy_to_matrix(pi_star) * env.R).sum(1)
P_pi = np.einsum("sa,sat->st", policy_to_matrix(pi_star), env.P)
v_check = np.linalg.solve(np.eye(25) - gamma * P_pi, r_pi)
assert np.allclose(v_star, v_check, atol=1e-4)
print(f"通过 ✅ 值迭代 {len(deltas)} 次收敛")
fig, ax = plt.subplots(1, 2, figsize=(9, 4))
env.plot(V=v_star, policy=pi_star, title="optimal v* and pi* (gamma=0.9)", ax=ax[0])
ax[1].semilogy(deltas); ax[1].set_title("||v_{k+1}-v_k||  (decays like gamma^k)"); ax[1].set_xlabel("iteration")
plt.tight_layout(); plt.show()

### 练习 2：数值验证压缩性

随机生成两个值向量 $\mathbf{v}_1,\mathbf{v}_2$，验证 $\|f(\mathbf{v}_1)-f(\mathbf{v}_2)\|_\infty\le\gamma\|\mathbf{v}_1-\mathbf{v}_2\|_\infty$。

In [ ]:
def check_contraction(env, gamma, n_trials=1000, seed=0):
    """返回所有试验中 ||f(v1)-f(v2)|| / ||v1-v2|| 的最大值"""
    rng = np.random.default_rng(seed)
    # TODO
    raise NotImplementedError

In [ ]:
#@title 💡 参考答案（点击展开查看；运行本单元即可使用参考实现） { display-mode: "form" }
def check_contraction(env, gamma, n_trials=1000, seed=0):
    """返回所有试验中 ||f(v1)-f(v2)|| / ||v1-v2|| 的最大值"""
    rng = np.random.default_rng(seed)
    worst = 0.0
    for _ in range(n_trials):
        v1, v2 = rng.normal(0, 10, env.n_states), rng.normal(0, 10, env.n_states)
        f1, _ = bellman_optimality_operator(env, v1, gamma)
        f2, _ = bellman_optimality_operator(env, v2, gamma)
        worst = max(worst, np.max(np.abs(f1 - f2)) / np.max(np.abs(v1 - v2)))
    return worst

In [ ]:
# ✅ 自测
ratio = check_contraction(env, gamma)
print(f"最大压缩比 {ratio:.4f} ≤ γ = {gamma}")
assert ratio <= gamma + 1e-9
print("通过 ✅")

---

## 5. 什么因素会影响最优策略？

最优策略由 $\gamma$ 和奖励共同决定。下面做两组实验（书中 3.5 节的例子）：

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
configs = [
    dict(gamma=0.9, r_forbidden=-1, title="gamma=0.9, r_forbidden=-1"),
    dict(gamma=0.5, r_forbidden=-1, title="gamma=0.5 (short-sighted)"),
    dict(gamma=0.0, r_forbidden=-1, title="gamma=0 (only immediate reward)"),
    dict(gamma=0.9, r_forbidden=-10, title="gamma=0.9, r_forbidden=-10"),
]
for ax, c in zip(axes, configs):
    e = GridWorld(r_forbidden=c["r_forbidden"])
    v, pi, _ = value_iteration(e, c["gamma"])
    e.plot(V=v, policy=pi, title=c["title"], ax=ax)
plt.tight_layout(); plt.show()

观察：

- $\gamma=0.9$、禁区惩罚 −1 时，**最优策略会穿过禁区**走捷径：走远路的 $\gamma^k$ 折扣损失比 −1 的惩罚更大。
- $\gamma=0.5$ 时，策略变得"短视"：远方目标的奖励被折扣得很厉害，**不值得为它承受禁区的 −1**，于是改为绕路；离目标远的格子，值趋近 0。
- $\gamma=0$ 时，只看即时奖励：只要不撞墙、不进禁区，所有动作都一样好（图中箭头只是 argmax 选了第一个）。
- 禁区惩罚变成 −10，策略就会**绕开禁区**。

> **对真机的启示**：奖励设计直接决定学到的行为。如果插线任务给"碰到线"一个小奖励（稠密奖励），策略可能学会"反复碰线刷分"而不去插入。这就是为什么 RLT/RECAP 等真机方法多用**稀疏的成功奖励**加人工干预。

---

## 小结

- 最优策略：所有状态的值都不低于任何其它策略。
- BOE：$v(s)=\max_a q(s,a)$；右边是压缩映射，所以解存在、唯一，并且迭代即可求得（值迭代）。
- 最优策略可取确定性的贪心策略 $\pi^*(s)=\arg\max_a q^*(s,a)$。
- $\gamma$ 与奖励设计共同决定最优行为。

**思考题**：如果把所有奖励都乘以 2，最优策略会变吗？加上一个常数呢？请改一下 `GridWorld` 的奖励参数，用代码验证你的猜想。